# 🦙 Llama 3 8B – Compliance Fine-Tuning (Google Colab)

**Before running:** Make sure your runtime is set to **T4 GPU** or higher.
Go to **Runtime → Change runtime type → T4 GPU** then click Save.

---
## Step 1: Install Dependencies

In [ ]:
%%capture
# Install Unsloth and essential dependencies
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl datasets transformers accelerate bitsandbytes

## Step 2: Upload & Extract Your Dataset

Upload your `dataset_2.1_rl.zip` file containing JSON documents.

In [ ]:
from google.colab import files
import os

print("Please upload your dataset_2.1_rl.zip file:")
uploaded = files.upload()

if not uploaded:
    print("❌ No file uploaded!")
else:
    zip_filename = list(uploaded.keys())[0]
    !unzip -q "{zip_filename}" -d .

    dataset_folder = "dataset_2.1_rl"
    if os.path.isdir(dataset_folder):
        file_count = len([f for f in os.listdir(dataset_folder) if f.endswith('.json')])
        print(f"✅ Dataset extracted! Found {file_count} JSON files in '{dataset_folder}'")
        DATA_DIR = dataset_folder
    else:
        print(f"⚠️ Could not find folder '{dataset_folder}'. Checking current directory...")
        file_count = len([f for f in os.listdir('.') if f.endswith('.json')])
        if file_count > 0:
            print(f"✅ Found {file_count} JSON files. Setting DATA_DIR to '.'")
            DATA_DIR = "."
        else:
            print("❌ No JSON files found.")

## Step 3: Configuration & Model Loading (Unsloth)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,
    load_in_4bit = LOAD_IN_4BIT,
)

print(f"✅ Loaded {MODEL_NAME}")

## Step 4: Dataset Preparation

In [ ]:
import json
from datasets import Dataset

def load_and_format_dataset(data_dir):
    conversations = []
    for filename in os.listdir(data_dir):
        if not filename.endswith(".json"):
            continue
        filepath = os.path.join(data_dir, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            instruction = data.get("instruction", "")
            user_input = data.get("input", "")
            final_text = data.get("output", {}).get("final_text", "")

            if not instruction or not final_text: continue

            convo = [
                {"role": "system", "content": "You are a compliance-focused insurance documentation assistant."},
                {"role": "user", "content": f"{instruction}\n\n{user_input}"},
                {"role": "assistant", "content": final_text}
            ]
            conversations.append({"messages": convo})
        except Exception as e:
            print(f"Error loading {filename}: {e}")

    return Dataset.from_list(conversations)

dataset = load_and_format_dataset(DATA_DIR)

def apply_chat_template(examples):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_chat_template, batched=True)
print(f"✅ Dataset ready with {len(dataset)} examples.")

## Step 5: Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", 
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ LoRA adapters applied.")

## Step 6: Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

## Step 7: Save Model & Inference

In [ ]:
OUTPUT_DIR = "compliance_llama3_model"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Model saved to {OUTPUT_DIR}")

# Inference Test
FastLanguageModel.for_inference(model)
messages = [
    {"role": "system", "content": "You are a compliance-focused insurance documentation assistant."},
    {"role": "user", "content": "Explain the importance of mortality charges in ULIPs logically and clearly."}
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=512, use_cache=True)
print("\n--- Model Output ---")
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))